# Preprocessing TF-IDF

Ce notebook décrit le preprocessing du texte pour l'apprentissage automatique avec TF-IDF :
nettoyage, normalisation, tokenisation/lemmatisation, puis vectorisation.
Il génère ensuite les matrices d'entraînement/validation et les enregistre pour la phase de modélisation.


In [1]:
from pathlib import Path
import json
import re
from typing import Iterable, Set

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from html import unescape
from unidecode import unidecode

import nltk
from nltk.corpus import stopwords

import spacy
from spacy.cli import download as spacy_download

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer

from scipy.sparse import save_npz


/Users/jimmydutto/Documents/GitHub/Rakuten/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Paramètres globaux pour la reproductibilité
RANDOM_STATE = 42
TEST_SIZE = 0.2
CLASS_WEIGHT = 'balanced'

ARTIFACTS_DIR = Path('../../artifacts/on_text/tfidf_baselines/v1')


In [3]:
# Localiser les CSV qui contiennent les textes d'entraînement et les labels
DATA_DIR = Path('../../Dataset')
X_train_path = DATA_DIR / 'X_train.csv'
Y_train_path = DATA_DIR / 'Y_train.csv'

# Lire les deux jeux de données en conservant l'index pour la fusion
X_train = pd.read_csv(X_train_path, index_col=0)
Y_train = pd.read_csv(Y_train_path, index_col=0)


In [4]:
# Fusionner features et labels, puis construire une colonne texte unique
train_df = X_train.join(Y_train, how='left')
text_columns = ['designation', 'description']

# Remplacer les valeurs manquantes par des chaînes vides pour éviter les NaN
train_df[text_columns] = train_df[text_columns].fillna('')

# Concaténer les champs textuels en une seule chaîne par produit
train_df['text_test'] = train_df[text_columns].agg(' '.join, axis=1)

train_df.head()


,designation,description,productid,imageid,prdtypecode,text_test
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,,3804725264,1263597046,10,Olivia: Personalisiertes Notizbuch / 150 Seite...
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,,436067568,1008141237,2280,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,Grand Stylet Ergonomique Bleu Gamepad Nintendo...
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,,50418756,457047496,1280,Peluche Donald - Europe - Disneyland 2000 (Mar...
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,La Guerre Des Tuques Luc a des id&eacute;es de...


In [5]:
# Fonctions utilitaires pour nettoyer le HTML, normaliser les espaces et standardiser le texte
def strip_html(text: str) -> str:
    # Supprimer les balises HTML en gardant des espacements lisibles.
    if not text:
        return ''
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text(separator=' ')


def normalize_whitespace(text: str) -> str:
    # Réduire les espaces/sauts de ligne multiples en un seul espace.
    return re.sub(r'\s+', ' ', text).strip()


def clean_text(text: str) -> str:
    # Chaîne complète de nettoyage appliquée avant la vectorisation.
    if text is None:
        text = ''
    text = unescape(text)  # Décoder les entités HTML comme &eacute;
    text = strip_html(text)
    text = text.lower()
    text = unidecode(text)  # Retirer les accents pour harmoniser les tokens
    text = normalize_whitespace(text)
    return text


In [6]:
# Télécharger les listes de stopwords et préparer le lemmatiseur spaCy
nltk.download('stopwords', quiet=True)

def normalize_stopword_list(words: Iterable[str]) -> Set[str]:
    # Retourner un ensemble normalisé de stopwords sans accents pour filtrage.
    return {unidecode(word).lower() for word in words}

french_stopwords = normalize_stopword_list(stopwords.words('french'))
english_stopwords = normalize_stopword_list(stopwords.words('english'))
stopword_set = french_stopwords | english_stopwords

def ensure_spacy_model(model_name: str) -> spacy.language.Language:
    # Charger un modèle spaCy, en le téléchargeant s'il n'est pas disponible.
    try:
        return spacy.load(model_name, disable=['ner'])
    except OSError:
        spacy_download(model_name)
        return spacy.load(model_name, disable=['ner'])

spacy_model = ensure_spacy_model('fr_core_news_sm')

def tokenize_and_lemmatize(text: str) -> Iterable[str]:
    # Tokeniser le texte nettoyé, filtrer le bruit et réduire aux lemmes.
    if not text:
        return []
    doc = spacy_model(text)
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and len(token) > 2
    ]
    filtered = [lemma for lemma in lemmas if lemma not in stopword_set]
    return filtered


In [7]:
# Configuration des filtres de fréquence pour le vocabulaire
APPLY_DF_FILTER = True
WORD_MIN_DF = 5
WORD_MAX_DF = 0.8
CHAR_MIN_DF = 5
CHAR_MAX_DF = 0.9


In [8]:
# TF-IDF au niveau des mots avec tokenisation lemmatisée (vocabulaire et bigrammes)
word_vectorizer = TfidfVectorizer(
    preprocessor=clean_text,
    tokenizer=tokenize_and_lemmatize,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=WORD_MIN_DF if APPLY_DF_FILTER else 1,
    max_df=WORD_MAX_DF if APPLY_DF_FILTER else 1.0,
    sublinear_tf=True,
)

# TF-IDF en n-grammes de caractères pour capter les sous-mots et les fautes
char_vectorizer = TfidfVectorizer(
    preprocessor=clean_text,
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=CHAR_MIN_DF if APPLY_DF_FILTER else 1,
    max_df=CHAR_MAX_DF if APPLY_DF_FILTER else 1.0,
    sublinear_tf=True,
)

# Combiner les features mots et caractères dans une représentation creuse unique
text_vectorizer = FeatureUnion([
    ('word', word_vectorizer),
    ('char', char_vectorizer),
])

# Envelopper l'union de features dans un pipeline qui sélectionne la colonne texte
text_pipeline = Pipeline([
    ('select_text', FunctionTransformer(lambda df: df['text_test'], validate=False)),
    ('vectorize', text_vectorizer),
])


In [9]:
# Séparer features/target et créer un split train/validation stratifié
X_features = train_df[['text_test']].copy()
y_target = train_df['prdtypecode'].copy()

X_train_split, X_valid_split, y_train_split, y_valid_split = train_test_split(
    X_features,
    y_target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_target,
)

# Ajuster le pipeline sur l'entraînement puis transformer les deux splits
vectorizer_model = text_pipeline.fit(X_train_split, y_train_split)

X_train_vectors = vectorizer_model.transform(X_train_split)
X_valid_vectors = vectorizer_model.transform(X_valid_split)

print(f'Train matrix shape: {X_train_vectors.shape}')
print(f'Validation matrix shape: {X_valid_vectors.shape}')


Train matrix shape: (67932, 317321)
Validation matrix shape: (16984, 317321)


In [10]:
# Sauvegarder les matrices creuses et les labels pour la phase de modélisation
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

save_npz(ARTIFACTS_DIR / 'X_train_vectors.npz', X_train_vectors)
save_npz(ARTIFACTS_DIR / 'X_valid_vectors.npz', X_valid_vectors)

np.save(ARTIFACTS_DIR / 'y_train.npy', y_train_split.to_numpy())
np.save(ARTIFACTS_DIR / 'y_valid.npy', y_valid_split.to_numpy())

label_names = sorted(y_target.unique().tolist())
(ARTIFACTS_DIR / 'label_names.json').write_text(
    json.dumps(label_names, ensure_ascii=False, indent=2)
)

metadata = {
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'class_weight': CLASS_WEIGHT,
    'word_min_df': WORD_MIN_DF if APPLY_DF_FILTER else 1,
    'word_max_df': WORD_MAX_DF if APPLY_DF_FILTER else 1.0,
    'char_min_df': CHAR_MIN_DF if APPLY_DF_FILTER else 1,
    'char_max_df': CHAR_MAX_DF if APPLY_DF_FILTER else 1.0,
}
(ARTIFACTS_DIR / 'metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2)
)

print(f'Saved artifacts to: {ARTIFACTS_DIR.resolve()}')


Saved artifacts to: /Users/jimmydutto/Documents/GitHub/Rakuten/artifacts/on_text/tfidf_baselines/v1
